In [2]:
import pandas as pd
import requests 
from bs4 import BeautifulSoup
import time

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

In [4]:
# Job title
titles = [
"data scientist",
"data analyst",
"machine learning engineer",
"AI engineer",
# "software engineer",
# "backend developer",
# "frontend developer",
# "full stack developer",
# "devops engineer",
# "cloud engineer" 
]

# Job location
location = "India"

# Starting point for pagination
page= 0

In [5]:
# Create an empty list to store job IDs

id_list = []
for title in titles:
    for page in range(0, 30, 10):  # 3 pages per title

        # Construct the URL for LinkedIn job search
        list_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={title.replace(' ', '+')}&location={location}&start={page}"

        # Send a GET request to the URL and store the response
        response = requests.get(list_url)

        # Get the HTML, parse the response and find all list items (job postings)
        list_data = response.text
        list_soup = BeautifulSoup(list_data, "html.parser")
        page_jobs = list_soup.find_all("li")
        print(f"{title} | page={page} | Found {len(page_jobs)} jobs")

        # Iterate through job postings to find job ids
        for job in page_jobs:
            try:
                base_card_div = job.find("div", {"class": "base-card"})
                job_id = base_card_div.get("data-entity-urn").split(":")[3]
                id_list.append({"search_title": title, "job_id": job_id})
            except:
                pass

        time.sleep(2)  # avoid getting blocked

print(f"\nTotal job IDs collected: {len(id_list)}")

data scientist | page=0 | Found 10 jobs
data scientist | page=10 | Found 10 jobs
data scientist | page=20 | Found 10 jobs
data analyst | page=0 | Found 10 jobs
data analyst | page=10 | Found 10 jobs
data analyst | page=20 | Found 10 jobs
machine learning engineer | page=0 | Found 10 jobs
machine learning engineer | page=10 | Found 10 jobs
machine learning engineer | page=20 | Found 10 jobs
AI engineer | page=0 | Found 10 jobs
AI engineer | page=10 | Found 10 jobs
AI engineer | page=20 | Found 10 jobs

Total job IDs collected: 120


In [6]:
 # Create an empty list to store job information
job_list = []

# Loop through the list of job IDs and get each URL
for item in id_list:
    
    # Construct the URL for each job using the job ID
    job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{item['job_id']}"

    # Send a GET request to the job URL and parse the response
    job_response = requests.get(job_url)
    job_soup = BeautifulSoup(job_response.text, "html.parser")

    # Create a dictionary to store job details
    job_post = {}

    # Store the search title
    job_post["search_title"] = item["search_title"]

    # Try to extract and store the job title
    try:
        job_post["job_title"] = job_soup.find("h2", {"class": "top-card-layout__title font-sans text-lg papabear:text-xl font-bold leading-open text-color-text mb-0 topcard__title"}).text.strip()
    except:
        job_post["job_title"] = None

    # Try to extract and store the company name
    try:
        job_post["company_name"] = job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"}).text.strip()
    except:
        job_post["company_name"] = None

    # Try to extract and store the location
    try:
        job_post["location"] = job_soup.find("span", {"class": "topcard__flavor topcard__flavor--bullet"}).text.strip()
    except:
        job_post["location"] = None

    # Try to extract and store the time posted
    try:
        job_post["time_posted"] = job_soup.find("span", {"class": "posted-time-ago__text topcard__flavor--metadata"}).text.strip()
    except:
        job_post["time_posted"] = None

    # Try to extract and store the number of applicants
    try:
        job_post["num_applicants"] = job_soup.find("span", {"class": "num-applicants__caption topcard__flavor--metadata topcard__flavor--bullet"}).text.strip()
    except:
        job_post["num_applicants"] = None

    # Append the job details to the job_list
    job_list.append(job_post)


In [7]:
# Check if the list contains all the desired data
job_list

[{'search_title': 'data scientist',
  'job_title': 'Data Scientist',
  'company_name': 'Deloitte',
  'location': 'Bengaluru, Karnataka, India',
  'time_posted': '4 days ago',
  'num_applicants': None},
 {'search_title': 'data scientist',
  'job_title': 'Data Scientist',
  'company_name': 'Reliance Retail',
  'location': 'Bangalore Urban, Karnataka, India',
  'time_posted': '1 week ago',
  'num_applicants': None},
 {'search_title': 'data scientist',
  'job_title': 'Data Scientist',
  'company_name': 'airtel',
  'location': 'Gurugram, Haryana, India',
  'time_posted': '3 weeks ago',
  'num_applicants': None},
 {'search_title': 'data scientist',
  'job_title': 'Data Scientist',
  'company_name': 'Deloitte',
  'location': 'Bengaluru, Karnataka, India',
  'time_posted': '2 days ago',
  'num_applicants': None},
 {'search_title': 'data scientist',
  'job_title': 'Machine Learning Engineer',
  'company_name': 'Meril',
  'location': 'Bengaluru, Karnataka, India',
  'time_posted': '2 weeks ago',

In [8]:
# Create a pandas DataFrame using the list of job dictionaries
jobs_df = pd.DataFrame(job_list)
jobs_df

,search_title,job_title,company_name,location,time_posted,num_applicants
0,data scientist,Data Scientist,Deloitte,"Bengaluru, Karnataka, India",4 days ago,NaN
1,data scientist,Data Scientist,Reliance Retail,"Bangalore Urban, Karnataka, India",1 week ago,NaN
2,data scientist,Data Scientist,airtel,"Gurugram, Haryana, India",3 weeks ago,NaN
3,data scientist,Data Scientist,Deloitte,"Bengaluru, Karnataka, India",2 days ago,NaN
4,data scientist,Machine Learning Engineer,Meril,"Bengaluru, Karnataka, India",2 weeks ago,NaN
...,...,...,...,...,...,...
115,AI engineer,Search Engineer - AI/ML,Apple,"Bengaluru, Karnataka, India",5 days ago,NaN
116,AI engineer,"Data Scientist, Research Cloud",Google,"Bengaluru, Karnataka, India",5 days ago,NaN
117,AI engineer,Machine Learning Engineer,Meril,"Chennai, Tamil Nadu, India",1 week ago,NaN
118,AI engineer,Data scientist,Infosys,"Bengaluru East, Karnataka, India",1 week ago,NaN


In [10]:
# Save data to CSV file

import csv
jobs_df.to_csv("india_linkedin_jobs.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_NONNUMERIC)
print(f"\nTotal jobs scraped: {len(jobs_df)}")
print("Saved to india_linkedin_jobs.csv")


Total jobs scraped: 120
Saved to india_linkedin_jobs.csv
